# clustering tiendas

In [81]:
import pandas as pd

In [82]:
df = pd.read_feather('data_dsmarket/df_preprocessed.feather')

In [83]:
print(df.columns.tolist())
print(df.shape)

['id', 'item', 'category', 'department', 'store', 'store_code', 'region', 'd', 'sales', 'date', 'weekday', 'event', 'yearweek', 'sell_price', 'season', 'ingresos', 'is_holiday']
(58327370, 17)


## FEATURE ENGINEERING

In [84]:
df_tiendas = df.groupby('store_code').agg(
    ventas_totales   = ('sales', 'sum'),
    precio_medio     = ('sell_price', 'mean'),
    ingresos_totales = ('ingresos', 'sum')
).reset_index()

info = df[['store_code', 'store', 'region']].drop_duplicates()
df_tiendas = df_tiendas.merge(info, on='store_code', how='left')

print(df_tiendas.shape)
df_tiendas

(10, 6)


,store_code,ventas_totales,precio_medio,ingresos_totales,store,region
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia


In [85]:
porcentajes_categorias = df.groupby(['store_code', 'category'])['sales'].sum().unstack(fill_value=0)
porcentajes_categorias = porcentajes_categorias.div(porcentajes_categorias.sum(axis=1), axis=0)
porcentajes_categorias

category,ACCESORIES,HOME_&_GARDEN,SUPERMARKET
store_code,,,
BOS_1,0.076687,0.248066,0.675248
BOS_2,0.088157,0.216687,0.695156
BOS_3,0.086587,0.229744,0.683669
NYC_1,0.113881,0.187149,0.698971
NYC_2,0.112181,0.275687,0.612132
NYC_3,0.085889,0.242349,0.671762
NYC_4,0.137549,0.175403,0.687049
PHI_1,0.127343,0.204894,0.667763
PHI_2,0.056573,0.214794,0.728633


In [86]:
df_tiendas = df_tiendas.merge(porcentajes_categorias, on='store_code', how='left')

In [87]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,0.113881,0.187149,0.698971
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,0.112181,0.275687,0.612132
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,0.137549,0.175403,0.687049
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,0.127343,0.204894,0.667763
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,0.056573,0.214794,0.728633
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,0.072602,0.191424,0.735974


In [88]:
transacciones = df[df['sales'] > 0].groupby('store_code')['ingresos'].sum()
ventas_con_compra = df[df['sales'] > 0].groupby('store_code')['sales'].sum()
df_tiendas['preciomedio'] = df_tiendas['store_code'].map(transacciones / ventas_con_compra)

In [89]:
porcentajes_estaciones = df.groupby(['store_code', 'season'])['sales'].sum().unstack(fill_value=0)
porcentajes_estaciones = porcentajes_estaciones.div(porcentajes_estaciones.sum(axis=1), axis=0)
porcentajes_estaciones

season,Invierno,Otoño,Primavera,Verano
store_code,,,,
BOS_1,0.244539,0.235342,0.269237,0.250883
BOS_2,0.245534,0.236748,0.265776,0.251942
BOS_3,0.244193,0.239163,0.265226,0.251418
NYC_1,0.243317,0.238041,0.266842,0.251801
NYC_2,0.247425,0.241674,0.261604,0.249298
NYC_3,0.241096,0.241038,0.263627,0.254240
NYC_4,0.245616,0.242869,0.267236,0.244279
PHI_1,0.265095,0.233043,0.265849,0.236013
PHI_2,0.261113,0.243208,0.255705,0.239974


In [90]:
df_tiendas = df_tiendas.merge(porcentajes_estaciones, on='store_code', how='left')

In [91]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,Invierno,Otoño,Primavera,Verano
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248,3.456737,0.244539,0.235342,0.269237,0.250883
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156,3.501995,0.245534,0.236748,0.265776,0.251942
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669,3.603713,0.244193,0.239163,0.265226,0.251418
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,0.113881,0.187149,0.698971,3.602647,0.243317,0.238041,0.266842,0.251801
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,0.112181,0.275687,0.612132,3.782838,0.247425,0.241674,0.261604,0.249298
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762,3.530222,0.241096,0.241038,0.263627,0.254240
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,0.137549,0.175403,0.687049,3.666525,0.245616,0.242869,0.267236,0.244279
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,0.127343,0.204894,0.667763,3.541098,0.265095,0.233043,0.265849,0.236013
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,0.056573,0.214794,0.728633,3.309360,0.261113,0.243208,0.255705,0.239974
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,0.072602,0.191424,0.735974,3.228417,0.258498,0.237018,0.267199,0.237286


In [92]:
ventas_totales = df.groupby('store_code')['sales'].sum()
ventas_festivos = df[df['is_holiday'] == 1].groupby('store_code')['sales'].sum()
df_tiendas['pct_festivos'] = (ventas_festivos / ventas_totales).values

In [93]:
df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)

ventas_finde = df[df['is_weekend'] == 1].groupby('store_code')['sales'].sum()

df_tiendas['pct_finde'] = (ventas_finde / ventas_totales).values

In [94]:
dias_totales = df.groupby('store_code')['date'].nunique()
dias_sin_ventas = df[df['sales'] == 0].groupby('store_code')['date'].nunique()

df_tiendas['tasa_dias_sin_venta'] = (dias_sin_ventas / dias_totales).values

In [95]:
ventas_inicio = df[df['date'].dt.year.isin([2011, 2012])].groupby('store_code')['sales'].sum()
ventas_final = df[df['date'].dt.year.isin([2015, 2016])].groupby('store_code')['sales'].sum()

df_tiendas['tendencia'] = (ventas_final / ventas_inicio).values

In [96]:
media_festivos = df[df['is_holiday'] == 1].groupby('store_code')['sales'].mean()
media_no_festivos = df[df['is_holiday'] == 0].groupby('store_code')['sales'].mean()

df_tiendas['ratio_festivos'] = (media_festivos / media_no_festivos).values

In [97]:
df_dispersion = df.groupby('store_code').agg(
    std_sales  = ('sales', 'std'),
    max_sales  = ('sales', 'max'),
    min_sales  = ('sales', 'min'),
    mean_sales = ('sales', 'mean')
).reset_index()

df_tiendas = df_tiendas.merge(df_dispersion, on='store_code', how='left')

In [98]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,Verano,pct_festivos,pct_finde,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248,3.456737,...,0.250883,0.034482,0.348351,1.0,0.815860,0.983997,3.327232,634,0,0.959291
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156,3.501995,...,0.251942,0.033891,0.346763,1.0,0.724142,0.966535,4.421267,626,0,1.236878
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669,3.603713,...,0.251418,0.032930,0.335612,1.0,0.917924,0.938192,3.796400,385,0,1.043992
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,0.113881,0.187149,0.698971,3.602647,...,0.251801,0.032348,0.362848,1.0,0.857044,0.921047,4.058652,648,0,1.319829
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,0.112181,0.275687,0.612132,3.782838,...,0.249298,0.032645,0.375393,1.0,0.912081,0.929789,2.759679,227,0,0.974753
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762,3.530222,...,0.254240,0.033267,0.337030,1.0,0.829562,0.948124,6.208486,763,0,1.918170
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,0.137549,0.175403,0.687049,3.666525,...,0.244279,0.032266,0.324281,1.0,0.922175,0.918633,2.004275,300,0,0.703559
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,0.127343,0.204894,0.667763,3.541098,...,0.236013,0.029037,0.360750,1.0,1.350346,0.823969,2.424396,224,0,0.882787
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,0.056573,0.214794,0.728633,3.309360,...,0.239974,0.029300,0.317730,1.0,1.323121,0.831645,3.866638,300,0,1.121945
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,0.072602,0.191424,0.735974,3.228417,...,0.237286,0.031266,0.341205,1.0,0.647428,0.889238,4.065281,374,0,1.102018


In [99]:
print(df_tiendas.shape)
print(df_tiendas.columns.tolist())
print(df_tiendas.isnull().sum())

(10, 23)
['store_code', 'ventas_totales', 'precio_medio', 'ingresos_totales', 'store', 'region', 'ACCESORIES', 'HOME_&_GARDEN', 'SUPERMARKET', 'preciomedio', 'Invierno', 'Otoño', 'Primavera', 'Verano', 'pct_festivos', 'pct_finde', 'tasa_dias_sin_venta', 'tendencia', 'ratio_festivos', 'std_sales', 'max_sales', 'min_sales', 'mean_sales']
store_code             0
ventas_totales         0
precio_medio           0
ingresos_totales       0
store                  0
region                 0
ACCESORIES             0
HOME_&_GARDEN          0
SUPERMARKET            0
preciomedio            0
Invierno               0
Otoño                  0
Primavera              0
Verano                 0
pct_festivos           0
pct_finde              0
tasa_dias_sin_venta    0
tendencia              0
ratio_festivos         0
std_sales              0
max_sales              0
min_sales              0
mean_sales             0
dtype: int64


In [100]:
df_tiendas

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,Verano,pct_festivos,pct_finde,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248,3.456737,...,0.250883,0.034482,0.348351,1.0,0.815860,0.983997,3.327232,634,0,0.959291
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156,3.501995,...,0.251942,0.033891,0.346763,1.0,0.724142,0.966535,4.421267,626,0,1.236878
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669,3.603713,...,0.251418,0.032930,0.335612,1.0,0.917924,0.938192,3.796400,385,0,1.043992
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,0.113881,0.187149,0.698971,3.602647,...,0.251801,0.032348,0.362848,1.0,0.857044,0.921047,4.058652,648,0,1.319829
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,0.112181,0.275687,0.612132,3.782838,...,0.249298,0.032645,0.375393,1.0,0.912081,0.929789,2.759679,227,0,0.974753
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762,3.530222,...,0.254240,0.033267,0.337030,1.0,0.829562,0.948124,6.208486,763,0,1.918170
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,0.137549,0.175403,0.687049,3.666525,...,0.244279,0.032266,0.324281,1.0,0.922175,0.918633,2.004275,300,0,0.703559
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,0.127343,0.204894,0.667763,3.541098,...,0.236013,0.029037,0.360750,1.0,1.350346,0.823969,2.424396,224,0,0.882787
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,0.056573,0.214794,0.728633,3.309360,...,0.239974,0.029300,0.317730,1.0,1.323121,0.831645,3.866638,300,0,1.121945
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,0.072602,0.191424,0.735974,3.228417,...,0.237286,0.031266,0.341205,1.0,0.647428,0.889238,4.065281,374,0,1.102018


In [103]:
numericas = [col for col in df_tiendas.columns if col not in ['store_code', 'store', 'region']]

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px

# Escalamos
X = df_tiendas[numericas].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

# Plots
fig_elbow = px.line(x=list(K), y=inertias, 
                    title='Elbow',
                    labels={'x': 'k', 'y': 'Inertia'})
fig_elbow.show()

fig_sil = px.line(x=list(K), y=silhouettes,
                  title='Silhouette',
                  labels={'x': 'k', 'y': 'Silhouette'})
fig_sil.show()

In [104]:
for k in [2, 3]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    df_tiendas[f'cluster_k{k}'] = kmeans.fit_predict(X_scaled)
    print(f"\nk={k}:")
    print(df_tiendas[['store_code', 'store', 'region', f'cluster_k{k}']])


k=2:
  store_code              store        region  cluster_k2
0      BOS_1          South_End        Boston           0
1      BOS_2            Roxbury        Boston           0
2      BOS_3           Back_Bay        Boston           0
3      NYC_1  Greenwich_Village      New York           0
4      NYC_2             Harlem      New York           0
5      NYC_3            Tribeca      New York           0
6      NYC_4           Brooklyn      New York           0
7      PHI_1    Midtown_Village  Philadelphia           1
8      PHI_2           Yorktown  Philadelphia           1
9      PHI_3      Queen_Village  Philadelphia           1

k=3:
  store_code              store        region  cluster_k3
0      BOS_1          South_End        Boston           2
1      BOS_2            Roxbury        Boston           2
2      BOS_3           Back_Bay        Boston           2
3      NYC_1  Greenwich_Village      New York           0
4      NYC_2             Harlem      New York           0
5 

In [111]:
for k in [2, 3]:
    print(f"\n{'='*50}")
    print(f"k={k}")
    print(f"{'='*50}")
    
    # Tiendas por cluster
    display(df_tiendas.groupby(f'cluster_k{k}')[['store_code', 'store', 'region']].agg(list))
    
    # Media de features por cluster
    print("\nMedia de features por cluster:")
    display(df_tiendas.groupby(f'cluster_k{k}')[numericas].mean().T)


k=2


,store_code,store,region
cluster_k2,,,
0,"[BOS_1, BOS_2, BOS_3, NYC_1, NYC_2, NYC_3, NYC_4]","[South_End, Roxbury, Back_Bay, Greenwich_Villa...","[Boston, Boston, Boston, New York, New York, N..."
1,"[PHI_1, PHI_2, PHI_3]","[Midtown_Village, Yorktown, Queen_Village]","[Philadelphia, Philadelphia, Philadelphia]"



Media de features por cluster:


/var/folders/nb/mmy0ln7n3w33lhzc3xlzr1zc0000gp/T/ipykernel_96572/3602452960.py:11: FutureWarning:

The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.



cluster_k2,0,1
ventas_totales,6.796365e+06,6.040285e+06
precio_medio,5.550624e+00,5.590997e+00
ingresos_totales,2.433351e+07,2.021379e+07
ACCESORIES,1.001329e-01,8.550582e-02
HOME_&_GARDEN,2.250119e-01,2.037042e-01
SUPERMARKET,6.748552e-01,7.107900e-01
Invierno,2.445311e-01,2.615685e-01
Otoño,2.392678e-01,2.377564e-01
Primavera,2.656495e-01,2.629175e-01
Verano,2.505515e-01,2.377577e-01



k=3


,store_code,store,region
cluster_k3,,,
0,"[NYC_1, NYC_2, NYC_4]","[Greenwich_Village, Harlem, Brooklyn]","[New York, New York, New York]"
1,"[PHI_1, PHI_2, PHI_3]","[Midtown_Village, Yorktown, Queen_Village]","[Philadelphia, Philadelphia, Philadelphia]"
2,"[BOS_1, BOS_2, BOS_3, NYC_3]","[South_End, Roxbury, Back_Bay, Tribeca]","[Boston, Boston, Boston, New York]"



Media de features por cluster:


/var/folders/nb/mmy0ln7n3w33lhzc3xlzr1zc0000gp/T/ipykernel_96572/3602452960.py:11: FutureWarning:

The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.



cluster_k3,0,1,2
ventas_totales,5.829122e+06,6.040285e+06,7.521796e+06
precio_medio,5.576571e+00,5.590997e+00,5.531163e+00
ingresos_totales,2.142914e+07,2.021379e+07,2.651179e+07
ACCESORIES,1.212035e-01,8.550582e-02,8.433002e-02
HOME_&_GARDEN,2.127462e-01,2.037042e-01,2.342112e-01
SUPERMARKET,6.660503e-01,7.107900e-01,6.814588e-01
Invierno,2.454525e-01,2.615685e-01,2.438402e-01
Otoño,2.408612e-01,2.377564e-01,2.380728e-01
Primavera,2.652272e-01,2.629175e-01,2.659663e-01
Verano,2.484592e-01,2.377577e-01,2.521208e-01


In [120]:
df_tiendas.sort_values(by = 'SUPERMARKET', ascending = False)

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,pct_finde,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,0.072602,0.191424,0.735974,3.228417,...,0.341205,1.0,0.647428,0.889238,4.065281,374,0,1.102018,1,1
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,0.056573,0.214794,0.728633,3.309360,...,0.317730,1.0,1.323121,0.831645,3.866638,300,0,1.121945,1,1
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,0.113881,0.187149,0.698971,3.602647,...,0.362848,1.0,0.857044,0.921047,4.058652,648,0,1.319829,0,0
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156,3.501995,...,0.346763,1.0,0.724142,0.966535,4.421267,626,0,1.236878,0,2
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,0.137549,0.175403,0.687049,3.666525,...,0.324281,1.0,0.922175,0.918633,2.004275,300,0,0.703559,0,0
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669,3.603713,...,0.335612,1.0,0.917924,0.938192,3.796400,385,0,1.043992,0,2
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248,3.456737,...,0.348351,1.0,0.815860,0.983997,3.327232,634,0,0.959291,0,2
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762,3.530222,...,0.337030,1.0,0.829562,0.948124,6.208486,763,0,1.918170,0,2
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,0.127343,0.204894,0.667763,3.541098,...,0.360750,1.0,1.350346,0.823969,2.424396,224,0,0.882787,1,1
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,0.112181,0.275687,0.612132,3.782838,...,0.375393,1.0,0.912081,0.929789,2.759679,227,0,0.974753,0,0


In [122]:
print(df_tiendas[numericas].std().sort_values(ascending=False))

ingresos_totales       6.742562e+06
ventas_totales         1.917808e+06
max_sales              1.993810e+02
std_sales              1.184961e+00
mean_sales             3.288007e-01
tendencia              2.315228e-01
ratio_festivos         5.293407e-02
SUPERMARKET            3.444760e-02
HOME_&_GARDEN          3.093911e-02
precio_medio           2.946683e-02
ACCESORIES             2.585617e-02
pct_finde              1.777553e-02
Invierno               8.534834e-03
Verano                 6.756005e-03
Primavera              3.830723e-03
Otoño                  3.373834e-03
pct_festivos           1.801817e-03
tasa_dias_sin_venta    0.000000e+00
min_sales              0.000000e+00
dtype: float64


/var/folders/nb/mmy0ln7n3w33lhzc3xlzr1zc0000gp/T/ipykernel_96572/2629921773.py:1: FutureWarning:

The default value of numeric_only in DataFrame.std is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.



In [128]:
numericas_revisadas = [
    'mean_sales',
    'std_sales', 
    'tendencia',
    'ratio_festivos',
    'SUPERMARKET',
    'HOME_&_GARDEN',
    'ACCESORIES',
    'precio_medio',
    'pct_finde',
]

In [129]:
X_clean = df_tiendas[numericas_revisadas].copy()
scaler = StandardScaler()
X_clean_scaled = scaler.fit_transform(X_clean)

# Elbow + Silhouette
inertias = []
silhouettes = []
K = range(2, 8)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_clean_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_clean_scaled, kmeans.labels_))

fig_elbow = px.line(x=list(K), y=inertias,
                    title='Elbow Curve',
                    labels={'x': 'k', 'y': 'Inertia'})
fig_elbow.show()

fig_sil = px.line(x=list(K), y=silhouettes,
                  title='Silhouette Score',
                  labels={'x': 'k', 'y': 'Silhouette'})
fig_sil.show()

In [133]:
for k in [2,3, 4]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    df_tiendas[f'cluster_k{k}'] = kmeans.fit_predict(X_clean_scaled)
    print(f"\nk={k}:")
    display(df_tiendas.groupby(f'cluster_k{k}')[['store_code', 'store', 'region']].agg(list))
    print("\nMedia de features por cluster:")
    display(df_tiendas.groupby(f'cluster_k{k}')[numericas_revisadas].mean().round(3).T)


k=2:


,store_code,store,region
cluster_k2,,,
0,"[BOS_1, BOS_2, BOS_3, NYC_1, NYC_3, NYC_4, PHI_3]","[South_End, Roxbury, Back_Bay, Greenwich_Villa...","[Boston, Boston, Boston, New York, New York, N..."
1,"[NYC_2, PHI_1, PHI_2]","[Harlem, Midtown_Village, Yorktown]","[New York, Philadelphia, Philadelphia]"



Media de features por cluster:


cluster_k2,0,1
mean_sales,1.183,0.993
std_sales,3.983,3.017
tendencia,0.816,1.195
ratio_festivos,0.938,0.862
SUPERMARKET,0.693,0.670
HOME_&_GARDEN,0.213,0.232
ACCESORIES,0.094,0.099
precio_medio,5.551,5.591
pct_finde,0.342,0.351



k=3:


...

cluster_k3,0,1,2
mean_sales,1.042,0.993,1.290
std_sales,3.376,3.017,4.438
tendencia,0.809,1.195,0.822
ratio_festivos,0.910,0.862,0.959
SUPERMARKET,0.707,0.670,0.681
HOME_&_GARDEN,0.185,0.232,0.234
ACCESORIES,0.108,0.099,0.084
precio_medio,5.577,5.591,5.531
pct_finde,0.343,0.351,0.342



k=4:


,store_code,store,region
cluster_k4,,,
0,"[NYC_1, NYC_4, PHI_3]","[Greenwich_Village, Brooklyn, Queen_Village]","[New York, New York, Philadelphia]"
1,"[NYC_2, PHI_1]","[Harlem, Midtown_Village]","[New York, Philadelphia]"
2,"[BOS_1, BOS_2, BOS_3, NYC_3]","[South_End, Roxbury, Back_Bay, Tribeca]","[Boston, Boston, Boston, New York]"
3,[PHI_2],[Yorktown],[Philadelphia]



Media de features por cluster:


cluster_k4,0,1,2,3
mean_sales,1.042,0.929,1.290,1.122
std_sales,3.376,2.592,4.438,3.867
tendencia,0.809,1.131,0.822,1.323
ratio_festivos,0.910,0.877,0.959,0.832
SUPERMARKET,0.707,0.640,0.681,0.729
HOME_&_GARDEN,0.185,0.240,0.234,0.215
ACCESORIES,0.108,0.120,0.084,0.057
precio_medio,5.577,5.584,5.531,5.605
pct_finde,0.343,0.368,0.342,0.318


In [140]:
df_tiendas[df_tiendas['cluster_k2'] == 0]

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3,cluster_k4
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248,3.456737,...,1.0,0.815860,0.983997,3.327232,634,0,0.959291,0,2,2
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156,3.501995,...,1.0,0.724142,0.966535,4.421267,626,0,1.236878,0,2,2
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669,3.603713,...,1.0,0.917924,0.938192,3.796400,385,0,1.043992,0,2,2
3,NYC_1,7698216,5.575351,27733958.0,Greenwich_Village,New York,0.113881,0.187149,0.698971,3.602647,...,1.0,0.857044,0.921047,4.058652,648,0,1.319829,0,0,0
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762,3.530222,...,1.0,0.829562,0.948124,6.208486,763,0,1.918170,0,2,2
6,NYC_4,4103676,5.574479,15046231.0,Brooklyn,New York,0.137549,0.175403,0.687049,3.666525,...,1.0,0.922175,0.918633,2.004275,300,0,0.703559,0,0,0
9,PHI_3,6427782,5.579928,20751558.0,Queen_Village,Philadelphia,0.072602,0.191424,0.735974,3.228417,...,1.0,0.647428,0.889238,4.065281,374,0,1.102018,0,0,0


In [141]:
df_tiendas[df_tiendas['cluster_k2'] == 1]

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3,cluster_k4
4,NYC_2,5685475,5.579885,21507232.0,Harlem,New York,0.112181,0.275687,0.612132,3.782838,...,1.0,0.912081,0.929789,2.759679,227,0,0.974753,1,1,1
7,PHI_1,5149062,5.588015,18233332.0,Midtown_Village,Philadelphia,0.127343,0.204894,0.667763,3.541098,...,1.0,1.350346,0.823969,2.424396,224,0,0.882787,1,1,1
8,PHI_2,6544012,5.605048,21656494.0,Yorktown,Philadelphia,0.056573,0.214794,0.728633,3.309360,...,1.0,1.323121,0.831645,3.866638,300,0,1.121945,1,1,3


In [139]:
df_tiendas[df_tiendas['cluster_k3'] == 2]

,store_code,ventas_totales,precio_medio,ingresos_totales,store,region,ACCESORIES,HOME_&_GARDEN,SUPERMARKET,preciomedio,...,tasa_dias_sin_venta,tendencia,ratio_festivos,std_sales,max_sales,min_sales,mean_sales,cluster_k2,cluster_k3,cluster_k4
0,BOS_1,5595292,5.525140,19341454.0,South_End,Boston,0.076687,0.248066,0.675248,3.456737,...,1.0,0.815860,0.983997,3.327232,634,0,0.959291,0,2,2
1,BOS_2,7214384,5.515620,25264736.0,Roxbury,Boston,0.088157,0.216687,0.695156,3.501995,...,1.0,0.724142,0.966535,4.421267,626,0,1.236878,0,2,2
2,BOS_3,6089330,5.541304,21944200.0,Back_Bay,Boston,0.086587,0.229744,0.683669,3.603713,...,1.0,0.917924,0.938192,3.796400,385,0,1.043992,0,2,2
5,NYC_3,11188180,5.542587,39496760.0,Tribeca,New York,0.085889,0.242349,0.671762,3.530222,...,1.0,0.829562,0.948124,6.208486,763,0,1.918170,0,2,2
